# colony_report

> Phase 18 (`specs/multiagent/phase18/`): the analysis of the colony-rule runs. Cells are levelled `<arm>|<fault>` (fault `clean`, `independent`, `collude` or `error`); values are paired on seed and team size. Exact one-sided sign-flip tests, Holm, the per-seed trend over log2 N, bootstrap intervals over seeds, the secondary measures (instruction-only control, share of the track record's benefit, goal drift, efficiency), a report and a figure (`colony_grid.syntheticRows18` makes test rows).

In [ ]:
#| default_exp colony_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Pairs and tests

In [ ]:
#| export
import json

import numpy as np

from explRL.bench_stats import holm

NAN = float('nan')


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return NAN


def cellVals(rows, arm, fault, metric, sizes=None):
    "{(seed, N): value} of one arm and fault (rows from CSV or memory)."
    out = {}
    for r in rows:
        if r['level'] == f'{arm}|{fault}' and (sizes is None or int(r['N']) in sizes):
            out[(int(r['seed']), int(r['N']))] = _f(r[metric])
    return out


def pairs(rows, arm, metric, fault='independent', base='plain', sizes=None):
    "{(seed, N): arm - base}, paired on seed and N (pairs missing either side are left out)."
    a, b = cellVals(rows, arm, fault, metric, sizes), cellVals(rows, base, fault, metric, sizes)
    return {k: a[k] - b[k] for k in sorted(a) if k in b and np.isfinite(a[k]) and np.isfinite(b[k])}


def signFlipExact(d, tail='greater', n=200_000, seed=0):
    "One-sided sign-flip p of the mean of d: exact over all 2**len(d) sign patterns up to 20 values, else seeded flips."
    d = np.asarray([x for x in d if np.isfinite(x)], float)
    if not len(d):
        return NAN
    obs, m = d.mean(), len(d)
    if m <= 20:
        signs = 1 - 2 * ((np.arange(2 ** m)[:, None] >> np.arange(m)) & 1)
        stats = (signs * d).mean(1)
    else:
        stats = (np.random.default_rng(seed).choice([-1.0, 1.0], size=(n, m)) * d).mean(1)
    return float((stats >= obs - 1e-12).mean() if tail == 'greater' else (stats <= obs + 1e-12).mean())


def bootSeeds(dp, stat=np.mean, nboot=2000, seed=0):
    "(estimate, lo, hi) of stat over the values of {(seed, N): v}, resampling seeds (a seed's pairs stay together)."
    if not dp:
        return NAN, NAN, NAN
    seeds = sorted({s for s, _ in dp})
    by = {s: [v for (s2, _), v in dp.items() if s2 == s] for s in seeds}
    rng, est = np.random.default_rng(seed), float(stat(list(dp.values())))
    bs = [stat([v for s in rng.choice(seeds, len(seeds)) for v in by[s]]) for _ in range(nboot)]
    return est, float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def trendSlopes(rows, arm, sizes, fault='independent', metric='honestPooled'):
    "{seed: least-squares slope over log2 N of arm - plain} for seeds with every size."
    dp, out = pairs(rows, arm, metric, fault, sizes=sizes), {}
    for s in sorted({s for s, _ in dp}):
        xs = [(np.log2(N), dp[(s, N)]) for N in sizes if (s, N) in dp]
        if len(xs) == len(sizes):
            out[s] = float(np.polyfit(*zip(*xs), 1)[0])
    return out


def lossRecovered(rows, arm, sizes):
    "1 - mean honest loss of arm / plain's (honest loss: welfareHonest clean - with independent liars; record: plain clean)."
    def hl(a):
        clean = cellVals(rows, 'plain' if a == 'record' else a, 'clean', 'welfareHonest', sizes)
        liar = cellVals(rows, a, 'independent', 'welfareHonest', sizes)
        return [clean[k] - liar[k] for k in liar if k in clean]
    base = np.mean(hl('plain')) if hl('plain') else NAN
    return float(1 - np.mean(hl(arm)) / base) if hl(arm) and base else NAN

## Summaries

In [ ]:
#| export
def _test(dp, nboot):
    est, lo, hi = bootSeeds(dp, nboot=nboot)
    return dict(n=len(dp), estimate=est, ci=[lo, hi], p=signFlipExact(list(dp.values())))


def efficiency(rows, arm, fault, sizes):
    "Team welfare per 1,000 output tokens and per dollar (LLM rows; NaN without usage)."
    t = [(_f(r['welfare']) * int(r['N']) * _f(r['tEnd']), _f(r.get('outputTokens')), _f(r.get('cost'))) for r in rows
         if r['level'] == f'{arm}|{fault}' and int(r['N']) in sizes]
    tot, tok, usd = (sum(x[i] for x in t) for i in range(3)) if t else (NAN, NAN, NAN)
    return dict(perKTokens=1000 * tot / tok if tok and np.isfinite(tok) else NAN, perDollar=tot / usd if usd and np.isfinite(usd) else NAN)


def colonySummaries(rows, plan, nboot=2000):
    "Every COLONY_PLAN item from Stage 1 rows (and Stage 2's, when present): hypotheses, Holm, secondary, mechanism."
    S1, m = plan['stage1']['sizes'], plan['margin']
    hyp = dict(
        H18_1=_test(pairs(rows, 'colony', 'honestPooled', sizes=S1), nboot),
        H18_2=_test(pairs(rows, 'plain', 'contamination', base='colony', sizes=S1), nboot),
        H18_3=_test({k: m + v for k, v in pairs(rows, 'colony', 'pooledShare', 'clean', sizes=S1).items()}, nboot),
        H18_4=_test(pairs(rows, 'colonyGuards', 'honestPooled', base='colony', sizes=S1), nboot))
    adj = holm([hyp[h]['p'] for h in plan['holm']])
    for h, a in zip(plan['holm'], adj):
        hyp[h].update(pHolm=float(a), supported=bool(a < plan['alpha'] and hyp[h]['estimate'] > 0))
    adv = pairs(rows, 'colony', 'honestPooled', sizes=S1)
    d5 = {(s, 0): adv[(s, S1[1])] - adv[(s, S1[0])] for s, _ in adv if (s, S1[0]) in adv and (s, S1[1]) in adv}
    hyp['H18_5'] = dict(_test(d5, nboot), stage=1, exploratory=True)
    hyp['H18_6'] = dict(_test(pairs(rows, 'colony', 'honestPooled', 'error', sizes=S1), nboot), secondary=True)
    arms, faults = plan['arms'], ('clean', 'independent', 'error', 'collude')
    per = {f'{a}|{f}': {N: {k: float(np.nanmean(list(v.values()))) if (v := cellVals(rows, a, f, k, [N])) else NAN
                            for k in plan['report']} for N in plan['sizes']}
           for a in arms for f in faults if any(r['level'] == f'{a}|{f}' for r in rows)}
    sec = dict(colonyVsConfirm=_test(pairs(rows, 'colony', 'honestPooled', base='confirm', sizes=S1), nboot),
               lossRecovered={a: lossRecovered(rows, a, S1) for a in arms},
               efficiency={a: efficiency(rows, a, 'independent', S1) for a in arms})
    lr = sec['lossRecovered']
    sec['shareOfRecord'] = {a: lr[a] / lr['record'] if lr.get('record') and np.isfinite(lr['record']) else NAN for a in arms}
    out = dict(hyp=hyp, secondary=sec, perCell=per, advantage={a: {N: bootSeeds(pairs(rows, a, 'honestPooled', sizes=[N]), nboot=nboot)
                                                               for N in plan['sizes']} for a in arms if a != 'plain'})
    S2 = plan.get('stage2', {})
    if S2.get('arm') and any(int(r['N']) in S2['sizes'] for r in rows):
        sl = trendSlopes(rows, S2['arm'], S1 + S2['sizes'])
        out['trend'] = dict(arm=S2['arm'], slopes=sl, estimate=float(np.mean(list(sl.values()))) if sl else NAN,
                            p=signFlipExact(list(sl.values())))
        out['collude'] = {N: bootSeeds(pairs(rows, S2['arm'], 'honestPooled', 'collude', sizes=[N]), nboot=nboot) for N in S2['sizes']}
    return out

## Report and figure

In [ ]:
#| export
def _fmt(x):
    return 'n/a' if x is None or not np.isfinite(x) else f'{x:.3f}'


def writeColony(path, S, meta):
    "report.md: hypotheses (estimate, interval, exact p, Holm), the advantage per N, secondary measures, goal drift."
    L = ['# Phase 18: colony rules (exploratory)', '', f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| hypothesis | n | estimate | 95% interval | p (one-sided, exact) | Holm | supported |', '|---|---|---|---|---|---|---|']
    for h, t in S['hyp'].items():
        L.append(f"| {h} | {t['n']} | {_fmt(t['estimate'])} | [{_fmt(t['ci'][0])}, {_fmt(t['ci'][1])}] | {_fmt(t['p'])} | "
                 f"{_fmt(t.get('pHolm'))} | {t.get('supported', 'secondary' if t.get('secondary') else 'exploratory')} |")
    L += ['', '## Advantage over the plain board (honest pooled share, independent liars)', '', '| arm | N | estimate | 95% interval |', '|---|---|---|---|']
    L += [f'| {a} | {N} | {_fmt(e)} | [{_fmt(lo)}, {_fmt(hi)}] |' for a, d in S['advantage'].items() for N, (e, lo, hi) in d.items()]
    sec = S['secondary']
    L += ['', '## Secondary', '', f"colony - confirm: {_fmt(sec['colonyVsConfirm']['estimate'])} (p {_fmt(sec['colonyVsConfirm']['p'])})", '',
          '| arm | loss recovered | share of the record\'s | welfare per 1k output tokens | per dollar |', '|---|---|---|---|---|']
    L += [f"| {a} | {_fmt(sec['lossRecovered'][a])} | {_fmt(sec['shareOfRecord'][a])} | {_fmt(sec['efficiency'][a]['perKTokens'])} | "
          f"{_fmt(sec['efficiency'][a]['perDollar'])} |" for a in sec['lossRecovered']]
    ks = sorted({k for d in S['perCell'].values() for v in d.values() for k in v})
    L += ['', '## Per arm, fault and N (means over seeds)', '', '| cell | N | ' + ' | '.join(ks) + ' |', '|---|---|' + '---|' * len(ks)]
    L += [f'| {c} | {N} | ' + ' | '.join(_fmt(v[k]) for k in ks) + ' |' for c, d in S['perCell'].items() for N, v in d.items()]
    if 'trend' in S:
        t = S['trend']
        L += ['', f"## Trend (H18-5 confirmatory, {t['arm']}): mean slope per log2 N {_fmt(t['estimate'])}, p {_fmt(t['p'])}"]
    open(path, 'w').write('\n'.join(L) + '\n')


def figColony(out, S):
    "colony.png: the advantage against N per arm (95% intervals), and contamination and goal drift per cell at the largest N."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8))
    for arm, d in S['advantage'].items():
        Ns = [N for N in d if np.isfinite(d[N][0])]
        if Ns:
            a.errorbar(Ns, [d[N][0] for N in Ns], yerr=[[d[N][0] - d[N][1] for N in Ns], [d[N][2] - d[N][0] for N in Ns]],
                       fmt='o-', capsize=3, label=arm)
    a.axhline(0, color='k', lw=0.6)
    a.set(xscale='log', xlabel='team size', ylabel='advantage over the plain board', title='Colony rules against team size')
    a.legend(fontsize=8)
    cells = [c for c in S['perCell'] if not c.endswith('|clean')]
    Nmax = max((N for d in S['perCell'].values() for N in d), default=None)
    x = np.arange(len(cells))
    for i, k in enumerate(('contamination', 'freeRiding')):
        b.bar(x + 0.4 * i, [S['perCell'][c][Nmax].get(k, NAN) for c in cells], 0.4, label=k)
    b.set_xticks(x + 0.2, cells, rotation=45, ha='right', fontsize=7)
    b.set(title=f'Safety and goal drift at N = {Nmax}')
    b.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{out}/colony.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

## Tests

In [ ]:
from explRL.colony_report import *
from explRL.colony_report import _f
from explRL.colony_grid import syntheticRows18
plan = dict(stage1=dict(sizes=[8, 16]), margin=0.05, holm=['H18_1', 'H18_2', 'H18_3', 'H18_4'], alpha=0.05,
            arms=['plain', 'confirm', 'colony', 'colonyGuards', 'record'], sizes=[8, 16],
            report=['honestPooled', 'contamination', 'freeRiding', 'misreport'])
S = colonySummaries(syntheticRows18(), plan, nboot=300)
h = S['hyp']
assert abs(h['H18_1']['estimate'] - 0.08 - 0.02 / 2) < 0.02 and h['H18_1']['n'] == 16      # gain + mean slope term
assert abs(h['H18_2']['estimate'] - 0.1) < 0.02 and abs(h['H18_3']['estimate'] - 0.06) < 0.02 and abs(h['H18_4']['estimate'] - 0.03) < 0.02
assert all(h[k]['supported'] for k in ('H18_1', 'H18_2', 'H18_3', 'H18_4')) and h['H18_1']['p'] == 1 / 2 ** 16
assert abs(h['H18_5']['estimate'] - 0.02) < 0.02 and h['H18_6']['n'] == 16
assert h['H18_1']['pHolm'] >= h['H18_1']['p']
# no effect: nothing supported
S0 = colonySummaries(syntheticRows18({a: (0, 0, 0, 0) for a in ('plain', 'confirm', 'colony', 'colonyGuards', 'record')}), plan, nboot=200)
assert not any(S0['hyp'][k]['supported'] for k in ('H18_1', 'H18_2', 'H18_4'))
# exact sign flip: all positive -> 1 / 2^n; symmetric -> large
assert signFlipExact([1, 2, 3]) == 1 / 8 and signFlipExact([1, -1]) == 0.75 and signFlipExact([-1, -2], 'less') == 0.25
# trend slopes recover the planted slope
rows = syntheticRows18(sizes=(8, 16, 32, 64), noise=0.0)
sl = trendSlopes(rows, 'colony', [8, 16, 32, 64])
assert len(sl) == 8 and all(abs(v - 0.02) < 1e-9 for v in sl.values())
plan2 = dict(plan, sizes=[8, 16, 32, 64], stage2=dict(arm='colony', sizes=[32, 64]))
S2 = colonySummaries(rows, plan2, nboot=100)
assert abs(S2['trend']['estimate'] - 0.02) < 1e-9 and S2['trend']['p'] == 1 / 256
# loss recovered: record's against plain clean; report and figure
assert np.isfinite(S['secondary']['lossRecovered']['record']) and np.isfinite(S['secondary']['efficiency']['colony']['perKTokens'])
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    writeColony(f'{d}/report.md', S, dict(mode='test'))
    figColony(d, S)
    assert os.path.getsize(f'{d}/colony.png') > 0 and 'H18_1' in open(f'{d}/report.md').read()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()